# Search Laboratory — A\* and Blind Search

**Artificial Intelligence — Laboratory Exercise: Search and A\***

This notebook completes every task in `Search.pdf` for the warehouse-robot navigation scenario:

* **Task 0** — formulate the search problem $\mathcal{P}=(S,A,T,s_0,G,c)$.
* **Task 1** — plan the agent (representation decisions).
* **Task 2** — implement A\* with the Manhattan heuristic.
* **Task 3** — systematic tests (original / trivial / no-solution / alternative paths).
* **Task 4** — inspect where each A\* concept lives in the code.
* **Task 5** — compare A\* with BFS (blind search).
* **Task 6** — investigate the heuristic ($h=0$, Euclidean, $2\times$).
* **Task 7 / Final Reflection** — evaluate the use of an LLM as an engineering tool.

> *Working output $\neq$ validated algorithm.* The code is treated as a hypothesis and tested
> against cases whose answers are known.


## The Warehouse Map

```
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
```

| Symbol | Meaning |
|---|---|
| `#` | Obstacle |
| `.` | Free cell |
| `S` | Start |
| `G` | Goal |

The robot moves **Up, Down, Left, Right**; every movement has cost 1.


In [1]:
WAREHOUSE = [
    "#################",
    "#S....#.........#",
    "#.###.#.#######.#",
    "#...#.#.......#.#",
    "###.#.#######.#.#",
    "#...#.........#.#",
    "#.###########.#.#",
    "#.............#G#",
    "#################",
]
for row in WAREHOUSE:
    print(row)
print("\nGrid:", len(WAREHOUSE), "rows x", len(WAREHOUSE[0]), "cols")


#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################

Grid: 9 rows x 17 cols


## Task 0 — Formulate the Search Problem

| Component | Specification for the warehouse |
|---|---|
| **State $S$** | A cell coordinate $(r,c)$ — the robot's position. |
| **Actions $A$** | $\{\text{Up},\text{Down},\text{Left},\text{Right}\}$ — move one cell orthogonally. |
| **Transition $T$** | $T((r,c),a)=(r+\Delta r,\,c+\Delta c)$ if that cell is in-bounds and not `#`; otherwise the action is undefined (not applicable). |
| **Initial state $s_0$** | The coordinate of `S`. |
| **Goal $G$** | $\{\,(r,c) : \text{cell}=G\,\}$ — here a single goal cell; goal test is $s=g$. |
| **Cost $c$** | $c(s,a,s')=1$ for every move (unit cost). |

**(a) What information specifies a state?** Just the robot's position $(r,c)$. The walls are
fixed and known, so they are part of the environment, not the state.

**(b) What makes an action invalid?** Moving off the grid, or into an obstacle cell `#`.

**(c) Is this deterministic?** Yes — each (state, action) pair yields exactly one successor.

**(d) What constitutes a solution?** A sequence of valid actions leading from $s_0$ to a goal
state. A *good* solution is a **shortest** such path (fewest unit-cost moves).


## Task 1 — Plan the Agent

1. **State representation:** a Python tuple `(row, col)`.
2. **Warehouse representation:** a list of strings; a helper builds a set of blocked cells.
3. **Valid actions:** the four orthogonal moves, filtered by bounds + obstacle checks.
4. **Goal recognition:** `state == goal`.
5. **Frontier contents:** a priority queue of entries `(f, g, state)` where `f = g + h`.
6. **Path reconstruction:** a `came_from` dict mapping each state to its parent.

**Reported on termination:** whether a solution was found, the path, the path length, and the
number of states expanded.


In [2]:
import heapq
import math
from collections import deque

def parse_map(grid):
    start = goal = None
    blocked = set()
    for r, line in enumerate(grid):
        for c, ch in enumerate(line):
            if ch == "S":
                start = (r, c)
            elif ch == "G":
                goal = (r, c)
            elif ch == "#":
                blocked.add((r, c))
    return start, goal, blocked, len(grid), len(grid[0])

MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # Up, Down, Left, Right

def neighbours(state, blocked, nrows, ncols):
    """Yield in-bounds, non-obstacle successors of `state` (the transition function T)."""
    r, c = state
    for dr, dc in MOVES:
        nr, nc = r + dr, c + dc
        if 0 <= nr < nrows and 0 <= nc < ncols and (nr, nc) not in blocked:
            yield (nr, nc)


## Task 2 — A\* Search (Manhattan heuristic)

The evaluation function is $f(n)=g(n)+h(n)$ where

$$g(n)=\text{cost from the start to } n, \qquad h(n)=|x-x_G|+|y-y_G|.$$

The frontier is a **min-priority queue** ordered by $f$. We avoid re-expanding states by keeping
a `best_g` record and skipping stale queue entries.


In [3]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def euclidean(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def astar(grid, heuristic=manhattan):
    """A* search. `heuristic(state, goal)` estimates remaining cost.

    Returns dict: found, path, length, expanded.
    """
    start, goal, blocked, nrows, ncols = parse_map(grid)

    g_start = 0
    h_start = heuristic(start, goal)
    frontier = [(g_start + h_start, g_start, start)]  # (f, g, state)
    came_from = {start: None}
    best_g = {start: 0}
    expanded = 0

    while frontier:
        f, g, current = heapq.heappop(frontier)
        if g > best_g.get(current, float("inf")):
            continue                      # stale entry -> skip
        expanded += 1

        if current == goal:               # goal test
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = came_from[node]
            path.reverse()
            return {"found": True, "path": path, "length": len(path) - 1,
                    "expanded": expanded, "start": start, "goal": goal}

        for nxt in neighbours(current, blocked, nrows, ncols):
            tentative_g = g + 1           # unit step cost
            if tentative_g < best_g.get(nxt, float("inf")):
                best_g[nxt] = tentative_g
                came_from[nxt] = current
                f_nxt = tentative_g + heuristic(nxt, goal)
                heapq.heappush(frontier, (f_nxt, tentative_g, nxt))

    return {"found": False, "path": [], "length": None,
            "expanded": expanded, "start": start, "goal": goal}

def bfs(grid):
    """Breadth-first (blind) search, for comparison with A*."""
    start, goal, blocked, nrows, ncols = parse_map(grid)
    frontier = deque([start])
    came_from = {start: None}
    expanded = 0
    while frontier:
        current = frontier.popleft()
        expanded += 1
        if current == goal:
            path = []
            node = current
            while node is not None:
                path.append(node)
                node = came_from[node]
            path.reverse()
            return {"found": True, "path": path, "length": len(path) - 1,
                    "expanded": expanded}
        for nxt in neighbours(current, blocked, nrows, ncols):
            if nxt not in came_from:
                came_from[nxt] = current
                frontier.append(nxt)
    return {"found": False, "path": [], "length": None, "expanded": expanded}


In [4]:
def render(grid, path):
    chars = [list(row) for row in grid]
    for (r, c) in path:
        if chars[r][c] not in ("S", "G"):
            chars[r][c] = "*"
    return "\n".join("".join(row) for row in chars)

res = astar(WAREHOUSE, manhattan)
print("A* on the warehouse map")
print("  found         :", res["found"])
print("  path length   :", res["length"])
print("  states expanded:", res["expanded"])
print("\nPath ('*'):\n")
print(render(WAREHOUSE, res["path"]))


A* on the warehouse map
  found         : True
  path length   : 40
  states expanded: 64

Path ('*'):

#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################


## Task 3 — Systematic Testing

A plausible path is not necessarily correct. Each test below has a *known* expected outcome.


In [5]:
def validate_path(grid, result):
    start, goal, blocked, nrows, ncols = parse_map(grid)
    if not result["found"]:
        return True
    path = result["path"]
    assert path[0] == start and path[-1] == goal
    for (r, c) in path:
        assert (r, c) not in blocked and 0 <= r < nrows and 0 <= c < ncols
    for p, q in zip(path, path[1:]):
        assert abs(p[0]-q[0]) + abs(p[1]-q[1]) == 1
    return True

# Test 1: original warehouse (already run above) -- revalidate.
print("Test 1  original : found =", res["found"], ", length =", res["length"],
      ", expanded =", res["expanded"], ", valid =", validate_path(WAREHOUSE, res))

# Test 2: trivial case -- goal adjacent to start -> one-step solution.
TRIVIAL = ["#####", "#SG##", "#####"]
t2 = astar(TRIVIAL)
print("Test 2  trivial  : found =", t2["found"], ", length =", t2["length"],
      "(expected 1), valid =", validate_path(TRIVIAL, t2))

# Test 3: no solution -- goal walled off -> must report failure, not loop.
NO_SOLUTION = ["#######", "#S....#", "###.###", "#...#G#", "#######"]
t3 = astar(NO_SOLUTION)
print("Test 3  no-path  : found =", t3["found"], "(expected False), expanded =",
      t3["expanded"])

# Test 4: alternative paths -- two routes of different length; must return the shortest.
ALT = [
    "#######",
    "#S...G#",
    "#.###.#",
    "#.....#",
    "#######",
]
t4 = astar(ALT)
print("Test 4  alt-paths: found =", t4["found"], ", length =", t4["length"],
      "(top route = 4 is shortest), valid =", validate_path(ALT, t4))
print("   returned path:", t4["path"])


Test 1  original : found = True , length = 40 , expanded = 64 , valid = True
Test 2  trivial  : found = True , length = 1 (expected 1), valid = True
Test 3  no-path  : found = False (expected False), expanded = 9
Test 4  alt-paths: found = True , length = 4 (top route = 4 is shortest), valid = True
   returned path: [(1, 1), (1, 2), (1, 3), (1, 4), (1, 5)]


## Task 4 — Where Each A\* Concept Appears

| Concept | Location in the code |
|---|---|
| **State** | the `(row, col)` tuple passed around the search. |
| **Action** | the offsets in `MOVES` used by `neighbours`. |
| **Transition** | `neighbours()` — generates valid successors (applies $T$). |
| **Goal test** | `if current == goal:` inside `astar`. |
| **$g(n)$** | `tentative_g = g + 1`, stored in `best_g`. |
| **$h(n)$** | `heuristic(nxt, goal)` (e.g. `manhattan`). |
| **$f(n)$** | `f_nxt = tentative_g + heuristic(nxt, goal)` — the priority key. |
| **Frontier** | the `heapq` min-heap `frontier` of `(f, g, state)`. |
| **Visited states** | `best_g` (best known cost) + the stale-entry skip. |
| **Path reconstruction** | the `came_from` dict and the back-walking loop. |

**(a) Frontier data structure?** A binary min-heap (priority queue) keyed on $f$.
**(b) How is the next state chosen?** `heapq.heappop` returns the frontier state with the
smallest $f=g+h$.
**(c) Where is the heuristic calculated?** In `manhattan`/`euclidean`, called when pushing a
successor.
**(d) Is $f=g+h$ explicit?** Yes — `f_nxt = tentative_g + heuristic(nxt, goal)`.
**(e) How is repeated exploration prevented?** `best_g` only accepts a successor if a strictly
cheaper $g$ is found, and popped entries with stale $g$ are skipped.


## Task 5 — Compare A\* with Blind Search (BFS)

In [6]:
a = astar(WAREHOUSE, manhattan)
b = bfs(WAREHOUSE)

print(f"{'Measure':<18}{'BFS':>8}{'A*':>8}")
print(f"{'Solution found':<18}{str(b['found']):>8}{str(a['found']):>8}")
print(f"{'Path length':<18}{b['length']:>8}{a['length']:>8}")
print(f"{'States expanded':<18}{b['expanded']:>8}{a['expanded']:>8}")


Measure                BFS      A*
Solution found        True    True
Path length             40      40
States expanded         64      64


**(a) Did both find a solution?** Yes.
**(b) Same length?** Yes — both are optimal on this unit-cost grid, so the path lengths match.
**(c) Which expanded fewer states?** On *this* map, neither — they expand the same number.
**(d) Why?** This warehouse is a **narrow maze**: it has only 64 traversable cells and essentially
one viable corridor, so *any* complete search must visit nearly every free cell before reaching
`G`. The Manhattan heuristic has almost no room to prune, so A\* and BFS expand the same set.
This is an honest and important result: **A\* is not automatically faster — its advantage depends
on the map leaving the heuristic room to discriminate between states.** The next cell shows a map
where A\* clearly wins.


### A more open map — where A\* actually helps

The warehouse maze hides A\*'s benefit. On an open grid with many alternative routes, the
Manhattan heuristic steers A\* toward the goal and it expands far fewer states than BFS, while
both still return a shortest path.


In [7]:
# S and G sit on the same row, close together, inside a tall open arena. BFS fans out in
# every direction and fills the arena; A* beelines along the heuristic gradient.
OPEN = [
    "###############",
    "#.............#",
    "#.............#",
    "#.............#",
    "#..S.......G..#",
    "#.............#",
    "#.............#",
    "#.............#",
    "###############",
]
oa = astar(OPEN, manhattan)
ob = bfs(OPEN)
print(f"{'Measure':<18}{'BFS':>8}{'A*':>8}")
print(f"{'Path length':<18}{ob['length']:>8}{oa['length']:>8}")
print(f"{'States expanded':<18}{ob['expanded']:>8}{oa['expanded']:>8}")
print("\nA* expands far fewer states here because the heuristic can discriminate "
      "between\ndirections on an open grid, even though both paths are shortest "
      f"({oa['length']} moves).")


Measure                BFS      A*
Path length              8       8
States expanded         65       9

A* expands far fewer states here because the heuristic can discriminate between
directions on an open grid, even though both paths are shortest (8 moves).


## Task 6 — Investigate the Heuristic

Manhattan distance is appropriate because the robot moves only horizontally/vertically with unit
cost: $|x-x_G|+|y-y_G|$ is exactly the length of an *unobstructed* path, so it never
overestimates the true remaining cost (it is **admissible** and, here, consistent). We now vary
the heuristic experimentally.


In [8]:
def scaled(h, factor):
    return lambda a, b: factor * h(a, b)

experiments = {
    "h = 0 (=> uniform-cost / Dijkstra)": lambda a, b: 0,
    "Manhattan (admissible)":             manhattan,
    "Euclidean (admissible, weaker)":     euclidean,
    "2 x Manhattan (inadmissible)":       scaled(manhattan, 2),
}

print(f"{'Heuristic':<38}{'found':>7}{'length':>8}{'expanded':>10}")
for name, h in experiments.items():
    r = astar(WAREHOUSE, h)
    print(f"{name:<38}{str(r['found']):>7}{str(r['length']):>8}{r['expanded']:>10}")


Heuristic                               found  length  expanded
h = 0 (=> uniform-cost / Dijkstra)       True      40        64
Manhattan (admissible)                   True      40        64
Euclidean (admissible, weaker)           True      40        64
2 x Manhattan (inadmissible)             True      40        67


### Interpretation

On this narrow-maze warehouse all three admissible variants expand the same 64 states, because
(as in Task 5) nearly every free cell must be visited regardless of the heuristic. The result is
still instructive:

* **$h=0$** turns A\* into uniform-cost search (Dijkstra). Optimal, no sense of direction. On an
  *open* map it would expand the most states; on this maze it matches the others.
* **Manhattan** is admissible and consistent, so A\* stays optimal (length 40). It is the
  strongest admissible choice — on open maps it prunes the most while remaining optimal.
* **Euclidean** is admissible but *weaker* on a 4-connected grid (it underestimates more because
  it allows for diagonal shortcuts that the robot cannot take); same optimal length here, and on
  open maps it typically expands somewhat more states than Manhattan.
* **$2\times$ Manhattan** is **inadmissible** (it can overestimate). Notice it expands **more**
  states here (67 > 64), not fewer: because the heuristic is no longer consistent, A\* re-opens
  nodes it had already settled when a cheaper $g$ is found later. More importantly it **loses the
  optimality guarantee** in general — on this map the returned length happens to stay 40, but that
  is luck, not a guarantee.

**Think About It — admissibility ($h\le h^*$):** while the heuristic stays optimistic
(admissible/consistent), A\* remains optimal and never re-expands a settled node. Once it becomes
too aggressive (inadmissible), A\* may trust an overestimate, commit early, re-expand nodes, and
return a sub-optimal path. "More intelligent" is not automatically "better" — it depends on the
*quality* of the information the heuristic provides.


## Task 7 — Evaluating the LLM-Generated Agent

1. **Correct immediately?** The core A\* loop (priority queue, $f=g+h$, reconstruction) is
   standard and usually correct.
2. **Bugs / design problems to watch for?** Marking a node "visited" on *generation* rather than
   on *expansion* can break optimality; forgetting the stale-entry skip wastes work; row/col vs
   x/y mix-ups in the heuristic.
3. **How discovered?** By the trivial and no-solution tests and by comparing A\* path length with
   BFS (both must agree when the heuristic is admissible).
4. **Unfamiliar terminology/structures?** `heapq` tuples for the priority queue; the `best_g`
   bookkeeping; admissibility/consistency.
5. **Modifications made?** Added the `best_g` stale-skip and an independent `validate_path`.
6. **Most useful tests?** No-solution (guards against infinite loops) and alternative-paths
   (guards against returning a non-shortest path).
7. **Trust without testing?** No — plausible output is not a validated algorithm.
8. **What was learned about A\*?** That optimality hinges on an admissible heuristic, and that
   the heuristic controls the *efficiency/optimality trade-off*, not just speed.


## Final Reflection

1. **Why formulate the search problem first?** Without clear states, actions, transitions, costs
   and a goal test, you cannot tell whether generated code solves the *right* problem — or test
   it meaningfully.
2. **In what sense is A\* "informed"?** It uses problem-specific knowledge $h(n)$ to estimate the
   remaining cost and prioritise promising states, unlike blind BFS/DFS.
3. **Why does the heuristic matter?** It determines both efficiency (how many states are
   expanded) and correctness (admissibility preserves optimality; an inadmissible heuristic can
   forfeit it).
4. **What did the LLM contribute?** A fast first implementation and explanations of library calls
   — boilerplate translation of a design into code.
5. **What could go wrong if accepted untested?** Subtle bugs (non-shortest paths, infinite loops
   on unsolvable maps, coordinate errors) that still "look like" working output.

> **AI Science → AI Engineering.** The scientific idea is *search*; A\* is one algorithm for it;
> the LLM is a tool to help implement it; the engineer remains responsible for understanding,
> testing and validating the result.
